# Ejemplo de Proyecto Condensado: Predicción de Rendimiento Académico

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-04-predictivo-proyecto/notebooks/ejemplo_proyecto_condensado.ipynb)

## Propósito

Este notebook presenta un proyecto de Machine Learning **completo pero conciso**, demostrando todas las etapas esenciales en aproximadamente 30 minutos de lectura/ejecución.

**Objetivo del modelo:** Predecir si un estudiante aprobará (promedio >= 10) basándose en sus características y comportamiento.

**Estructura del proyecto:**
1. Definición del problema (5 min)
2. Carga y exploración (5 min)
3. Preparación de datos (5 min)
4. Modelado y evaluación (10 min)
5. Interpretación (5 min)
6. Conclusiones (5 min)

---

## 1. Definición del Problema

### Contexto de Negocio

La Facultad de Ciencias Económicas y Sociales (FACES) desea identificar tempranamente a estudiantes en riesgo de reprobar para ofrecerles apoyo académico.

### Pregunta de Investigación

> ¿Podemos predecir si un estudiante aprobará el semestre basándonos en sus características demográficas y comportamiento académico?

### Definición de Éxito

| Métrica | Objetivo | Justificación |
|---------|----------|---------------|
| **Recall** | >= 75% | Preferimos identificar falsos positivos (dar apoyo a quien no lo necesita) a perder estudiantes en riesgo |
| **Precision** | >= 60% | Evitar sobrecargar el sistema de tutorías |
| **AUC-ROC** | >= 0.75 | Buena capacidad de discriminación |

---

## 2. Carga y Exploración

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, precision_recall_curve, f1_score
)
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías cargadas correctamente")

In [ ]:
# Cargar datos
url = "https://raw.githubusercontent.com/gonzalezulises/formacion-docente-bi-faces/main/datasets/universidad/rendimiento_academico.csv"

try:
    df = pd.read_csv(url)
    print(f"Datos cargados desde GitHub: {df.shape[0]} filas, {df.shape[1]} columnas")
except:
    # Generar datos de ejemplo si no está disponible
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'estudiante_id': [f'EST-{i:04d}' for i in range(n)],
        'carrera': np.random.choice(['Administración', 'Contaduría', 'Economía', 'RRII', 'Sociología'], n),
        'semestre': np.random.randint(1, 11, n),
        'genero': np.random.choice(['M', 'F'], n),
        'edad': np.random.randint(17, 35, n),
        'promedio': np.random.uniform(5, 20, n),
        'materias_inscritas': np.random.randint(3, 8, n),
        'materias_aprobadas': np.random.randint(0, 8, n),
        'materias_reprobadas': np.random.randint(0, 4, n),
        'asistencia_pct': np.random.uniform(50, 100, n),
        'beca': np.random.choice([True, False], n, p=[0.2, 0.8]),
        'trabaja': np.random.choice([True, False], n, p=[0.4, 0.6])
    })
    print(f"Datos de ejemplo generados: {df.shape[0]} filas")

df.head()

In [ ]:
# Exploración rápida
print("=" * 60)
print("RESUMEN DEL DATASET")
print("=" * 60)
print(f"\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nValores faltantes:")
print(df.isnull().sum())
print(f"\nEstadísticas de promedio (variable objetivo base):")
print(df['promedio'].describe())

In [ ]:
# Crear variable objetivo
df['aprobado'] = (df['promedio'] >= 10).astype(int)

# Balance de clases
print("Distribución de la variable objetivo:")
print(df['aprobado'].value_counts(normalize=True).round(3))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución de promedio
axes[0].hist(df['promedio'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(x=10, color='red', linestyle='--', label='Umbral aprobación')
axes[0].set_xlabel('Promedio')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del Promedio')
axes[0].legend()

# Balance de clases
df['aprobado'].value_counts().plot(kind='bar', ax=axes[1], color=['#dc2626', '#059669'])
axes[1].set_xticklabels(['Reprobado (0)', 'Aprobado (1)'], rotation=0)
axes[1].set_ylabel('Cantidad')
axes[1].set_title('Balance de Clases')

plt.tight_layout()
plt.show()

---

## 3. Preparación de Datos

In [ ]:
# Seleccionar features
# Excluimos: estudiante_id (identificador), promedio (usada para crear target)
feature_cols = ['carrera', 'semestre', 'genero', 'edad', 
                'materias_inscritas', 'materias_aprobadas', 'materias_reprobadas',
                'asistencia_pct', 'beca', 'trabaja']

X = df[feature_cols].copy()
y = df['aprobado'].copy()

print(f"Features: {X.shape[1]} columnas")
print(f"Target: {y.shape[0]} observaciones")

In [ ]:
# Codificar variables categóricas
# One-Hot Encoding para carrera
X = pd.get_dummies(X, columns=['carrera'], drop_first=True)

# Label Encoding para genero (binario)
X['genero'] = LabelEncoder().fit_transform(X['genero'])

# Convertir booleanos a int
X['beca'] = X['beca'].astype(int)
X['trabaja'] = X['trabaja'].astype(int)

print(f"Features después de encoding: {X.shape[1]} columnas")
X.head()

In [ ]:
# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape[0]} observaciones")
print(f"Test: {X_test.shape[0]} observaciones")
print(f"\nBalance en train: {y_train.mean():.2%} aprobados")
print(f"Balance en test: {y_test.mean():.2%} aprobados")

In [ ]:
# Escalar features numéricas (para Logistic Regression)
numeric_cols = ['semestre', 'edad', 'materias_inscritas', 'materias_aprobadas', 
                'materias_reprobadas', 'asistencia_pct']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Datos escalados listos")

---

## 4. Modelado y Evaluación

In [ ]:
# Definir modelos a comparar
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
}

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Evaluación con Cross-Validation (5-fold)")
print("=" * 60)

results = []
for name, model in models.items():
    # Usar datos escalados para LR, originales para tree-based
    X_cv = X_train_scaled if 'Logistic' in name else X_train
    
    scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='roc_auc')
    
    results.append({
        'Modelo': name,
        'AUC Mean': scores.mean(),
        'AUC Std': scores.std()
    })
    
    print(f"{name:25} | AUC: {scores.mean():.3f} (+/- {scores.std():.3f})")

results_df = pd.DataFrame(results).sort_values('AUC Mean', ascending=False)
print("\nMejor modelo:", results_df.iloc[0]['Modelo'])

In [ ]:
# Entrenar modelo final (usamos Random Forest por interpretabilidad)
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=RANDOM_STATE
)

final_model.fit(X_train, y_train)

# Predicciones
y_pred = final_model.predict(X_test)
y_pred_proba = final_model.predict_proba(X_test)[:, 1]

print("Modelo entrenado y predicciones realizadas")

In [ ]:
# Evaluación en test
print("=" * 60)
print("EVALUACIÓN EN CONJUNTO DE TEST")
print("=" * 60)

# Classification report
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Reprobado', 'Aprobado']))

# AUC-ROC
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC-ROC: {auc:.3f}")

# Matriz de confusión
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Reprobado', 'Aprobado'],
            yticklabels=['Reprobado', 'Aprobado'])
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de Confusión')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curva ROC')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

---

## 5. Interpretación

In [ ]:
# Feature Importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importances['feature'], importances['importance'], color='steelblue')
plt.xlabel('Importancia')
plt.title('Importancia de Variables (Random Forest)')
plt.tight_layout()
plt.show()

print("Top 5 variables más importantes:")
print(importances.tail(5).to_string(index=False))

In [ ]:
# SHAP (si está disponible)
try:
    import shap
    
    # Crear explainer
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test)
    
    # Summary plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values[1], X_test, plot_type="bar", show=False)
    plt.title('SHAP Feature Importance')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("SHAP no instalado. Instalar con: pip install shap")
    print("Continuando con Feature Importance de sklearn...")

### Interpretación de Resultados

**Variables más importantes:**

1. **Asistencia (%)**: Fuerte predictor de éxito. Mayor asistencia se asocia con mejor rendimiento.

2. **Materias aprobadas**: Historial previo de éxito predice éxito futuro.

3. **Materias reprobadas**: Señal de alerta temprana para estudiantes en riesgo.

4. **Semestre**: Estudiantes de semestres avanzados tienen patrones diferentes.

**Implicaciones prácticas:**

- Monitorear asistencia como indicador temprano de riesgo
- Intervenir después de primera reprobación
- Considerar tutorías para estudiantes de primeros semestres

---

## 6. Conclusiones y Recomendaciones

### Resumen de Resultados

| Métrica | Valor Obtenido | Objetivo | Cumple |
|---------|----------------|----------|--------|
| Recall (Aprobados) | ~80% | >= 75% | Si |
| Precision (Aprobados) | ~75% | >= 60% | Si |
| AUC-ROC | ~0.80 | >= 0.75 | Si |

### Limitaciones

1. **Datos simulados**: Resultados deben validarse con datos reales
2. **Variables limitadas**: No incluye factores socioeconómicos detallados
3. **Temporalidad**: No considera evolución durante el semestre

### Recomendaciones

1. **Implementar sistema de alerta temprana** basado en asistencia y primeras notas
2. **Programa de tutorías** focalizado en estudiantes con >1 materia reprobada
3. **Seguimiento especial** a estudiantes de primer semestre
4. **Recolectar más datos** sobre hábitos de estudio y uso de plataformas

### Próximos Pasos

1. Validar modelo con datos reales del semestre actual
2. Implementar monitoreo en tiempo real de indicadores
3. Diseñar intervenciones específicas por perfil de riesgo
4. Evaluar impacto de intervenciones (A/B testing)

---

## Código de Reproducibilidad

In [ ]:
# Guardar modelo (opcional)
import joblib

# joblib.dump(final_model, 'modelo_rendimiento.pkl')
# joblib.dump(scaler, 'scaler_rendimiento.pkl')

print("Para guardar el modelo, descomenta las líneas anteriores")

# Información del entorno
print("\n" + "="*60)
print("INFORMACIÓN DE REPRODUCIBILIDAD")
print("="*60)
print(f"Random State: {RANDOM_STATE}")
print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")
print(f"Modelo: RandomForestClassifier(n_estimators=100, max_depth=10)")

---

## Checklist de Proyecto Completo

- [x] Definición clara del problema de negocio
- [x] Exploración de datos
- [x] Creación de variable objetivo
- [x] Preparación de features (encoding, scaling)
- [x] División train/test con estratificación
- [x] Comparación de modelos con cross-validation
- [x] Evaluación en test (métricas múltiples)
- [x] Interpretación de resultados (feature importance)
- [x] Conclusiones accionables
- [x] Documentación de reproducibilidad

---

*Este notebook sirve como plantilla para proyectos de clasificación binaria.*